les horodatages manquants:

In [11]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("data/dataset_propre.csv")
data["DATE_TIME"] = pd.to_datetime(data["DATE_TIME"])
data = data.sort_values("DATE_TIME").reset_index(drop=True)
data.head()

,DATE_TIME,AC_POWER_TOTAL,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,0.0,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,0.0,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,0.0,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,0.0,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,0.0,24.621525,22.165423,0.0


Localiser les trous:

In [12]:
# Écart entre chaque horodatage et le précédent
data["ecart"] = data["DATE_TIME"].diff()

# Tout ce qui n'est pas 15 minutes = un trou
trous = data[data["ecart"] > pd.Timedelta("15min")]
print("Nombre de trous :", len(trous))
print(trous[["DATE_TIME", "ecart"]])

Nombre de trous : 10
               DATE_TIME           ecart
93   2020-05-16 02:00:00 0 days 03:00:00
420  2020-05-19 12:30:00 0 days 01:00:00
520  2020-05-20 17:30:00 0 days 04:15:00
542  2020-05-21 07:45:00 0 days 09:00:00
724  2020-05-23 06:45:00 0 days 01:45:00
784  2020-05-23 22:00:00 0 days 00:30:00
911  2020-05-25 06:00:00 0 days 00:30:00
1265 2020-05-29 06:15:00 0 days 08:00:00
1776 2020-06-03 14:15:00 0 days 00:30:00
3088 2020-06-17 06:45:00 0 days 00:45:00


Reconstruire une grille temporelle complète:

In [13]:
# Grille régulière de 15 min, du début à la fin
grille = pd.date_range(
    start=data["DATE_TIME"].min(),
    end=data["DATE_TIME"].max(),
    freq="15min"
)

# Réindexer sur cette grille
data_complet = data.set_index("DATE_TIME").reindex(grille)
data_complet.index.name = "DATE_TIME"

print("Lignes avant :", len(data))
print("Lignes après :", len(data_complet))
print("\nValeurs manquantes créées :")
print(data_complet.isna().sum())

Lignes avant : 3157
Lignes après : 3264

Valeurs manquantes créées :
AC_POWER_TOTAL         107
AMBIENT_TEMPERATURE    107
MODULE_TEMPERATURE     107
IRRADIATION            107
ecart                  108
dtype: int64


Combler les trous:

In [14]:
# Interpolation linéaire pour les trous courts
colonnes = ["AC_POWER_TOTAL", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]
data_complet[colonnes] = data_complet[colonnes].interpolate(method="linear", limit=4)

print("Restant après interpolation :")
print(data_complet.isna().sum())

Restant après interpolation :
AC_POWER_TOTAL          79
AMBIENT_TEMPERATURE     79
MODULE_TEMPERATURE      79
IRRADIATION             79
ecart                  108
dtype: int64


Sur un pas de 15 minutes, l’irradiance et la production varient de façon continue, donc l’interpolation linéaire est physiquement défendable sur de courtes durées. Sur plusieurs heures, non.

In [15]:
manquants = data_complet[data_complet["AC_POWER_TOTAL"].isna()]
print(manquants.index.hour.value_counts().sort_index())

DATE_TIME
0     11
1     12
2      8
3      8
4      8
5      8
6      7
7      3
14     2
15     4
16     4
17     2
23     2
Name: count, dtype: int64


In [16]:
data_complet = data_complet.drop(columns=["ecart"])
data_complet.to_csv("data/dataset_regulier.csv")

**Traitement des 79 valeurs manquantes restantes.** Ces lignes n'ont pas été
traitées explicitement : elles sont écartées automatiquement lors du filtrage
des heures de jour (`IRRADIATION > 0.01`), une comparaison avec NaN étant
évaluée à False par pandas. Le jeu de données final ne contient donc aucune
valeur manquante, ce qui a été vérifié.

Sur ces 79 points, environ 67 étaient nocturnes (heures 0-5 et 23) et 12
diurnes — leur exclusion représente moins de 0,8 % des pas de temps
exploitables.

Filtrer les heures de nuit:

In [17]:
data = pd.read_csv("data/dataset_regulier.csv", index_col="DATE_TIME", parse_dates=True)

# Garder uniquement les heures où la centrale peut produire
jour = data[data["IRRADIATION"] > 0.01].copy()

print("Avant :", len(data))
print("Après :", len(jour))
print("Proportion conservée :", round(100 * len(jour) / len(data), 1), "%")

Avant : 3264
Après : 1652
Proportion conservée : 50.6 %


Vérifier que la production résiduelle est cohérente


In [18]:
print(jour["AC_POWER_TOTAL"].describe())

# Combien de points à production nulle malgré du soleil ?
print("Zéros restants :", (jour["AC_POWER_TOTAL"] == 0).sum())

count     1652.000000
mean     12905.829402
std       7920.812875
min          0.000000
25%       5684.860268
50%      13307.511607
75%      19120.155357
max      29150.212499
Name: AC_POWER_TOTAL, dtype: float64
Zéros restants : 2


Les stats sont saines : 50,6% conservé (cohérent avec un cycle jour/nuit), moyenne ~12 900 kW, médiane ~13 300 kW, max 29 150 kW. Une distribution assez symétrique, ce qui est bon signe pour la modélisation.

2 points à production nulle malgré une irradiance non nulle ont été exclus — indisponibilité d'équipement plutôt que comportement physique normal. Cible du modèle = centrale en fonctionnement nominal.

In [20]:
jour = jour[jour["AC_POWER_TOTAL"] > 0].copy()
print("Lignes finales :", len(jour))
jour.to_csv("data/dataset_jour.csv")

Lignes finales : 1650
